In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
    "./test/modules/whisper_streaming"
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from pathlib import Path
import librosa
import time

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import search_all_ref_and_hyp
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *
from sj_utils.evaluator import TimeChecker

In [ ]:
from whisper_online import FasterWhisperASR, OnlineASRProcessor

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/test/"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000

In [ ]:
src = Path(SOURCE)

In [ ]:
transcribe_time = TimeChecker()
processed_time = TimeChecker()

In [ ]:
asr = FasterWhisperASR("en", MODEL_SIZE)
asr.use_vad()
online = OnlineASRProcessor(asr)

In [ ]:
def normalize_text(text):
    return normalize_text_only_en(text).upper()

In [ ]:
def transcriber(flac:Path) -> TRNFormat:
    audio, _ = librosa.load(flac, sr=SAMPLE_RATE)
    online.init()

    print(f"Processing")
    print(f"\tAudio name: {flac.name}")
    print(f"\tAudio length: {len(audio) / SAMPLE_RATE:.2f} seconds")

    full_text = ""
    for segment in segment_audio(audio):
        transcribe_time.start()
        online.insert_audio_chunk(segment)
        _, _, text = online.process_iter()
        transcribe_time.check()
        full_text += text
    _, _, text = online.finish()
    full_text += text
    text = normalize_text(full_text)

    return TRNFormat(id = flac.stem, text = text)

In [ ]:
processed_time.start()
data = search_all_ref_and_hyp(src, transcriber, normalize_text, 2)
processed_time.check()

In [ ]:
concat_result = {}
for value in data.values():
    for k, v in value.items():
        if k not in concat_result:
            concat_result[k] = []
        concat_result[k].extend(v)

In [ ]:
output = sclite_trn(
    concat_result["ref"],
    concat_result["hyp"],
)

In [ ]:
{
    "result":parse_sclite_summary(output),
    "processed_time": processed_time.metric(),
    "transcribe_time": transcribe_time.metric(),
}